# Teach MiniT2I a New Style with LoRA

MiniT2I is a text-to-image model: give it a text prompt and it generates a matching picture. This notebook adapts MiniT2I-B/16 to a new visual style — the look of the *Naruto* anime — by finetuning it on the [`lambdalabs/naruto-blip-captions`](https://huggingface.co/datasets/lambdalabs/naruto-blip-captions) dataset of anime images and captions. Afterwards you can type your own prompts and get images in that style.

Rather than retraining the whole model (hundreds of millions of weights), we use **LoRA** (Low-Rank Adaptation). LoRA freezes the original model and trains a small set of extra "adapter" weights alongside it. This is fast, fits in a single GPU's memory, and produces a tiny adapter file you can share and load on top of the base model.

By the end you will have:
- a trained LoRA adapter saved to disk,
- validation images showing the style emerging during training,
- fresh samples generated from your own prompts.

**Before you start:** switch Colab to a GPU runtime via **Runtime > Change runtime type > Hardware accelerator > GPU**. The full run trains for 400 steps and takes roughly 15–25 minutes on a typical Colab GPU. To try the pipeline end to end more quickly, set `MAX_TRAIN_STEPS = 50` in the configuration cell.

Everything needed to run is contained in this single notebook — no repository checkout required.

Links: [GitHub](https://github.com/Hope7Happiness/minit2i-torch) · [Hugging Face](https://huggingface.co/MiniT2I/MiniT2I)

# 1. Setup

Install the Python packages this notebook needs. This runs once per Colab session and takes about a minute; the output is hidden to keep things tidy.

In [ ]:
%%capture
!pip install -U -q diffusers transformers datasets accelerate peft safetensors huggingface_hub matplotlib
!pip uninstall -y -q torchao

In [ ]:
#@title Runtime setup
import os

os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import random
import json
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU runtime."
device = torch.device("cuda")
weight_dtype = torch.bfloat16
print("GPU:", torch.cuda.get_device_name(0))

# 2. Sign in to Hugging Face

The MiniT2I weights are hosted on Hugging Face and require you to be signed in to download them. You will need a Hugging Face account and an access token (create one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)).

The easiest way in Colab: click the key icon in the left sidebar, add a secret named `HF_TOKEN` with your token as the value, and enable notebook access. If no secret is found, this cell opens a login box where you can paste the token instead.

If this cell reports that it cannot access the model, your token does not have permission for the `MiniT2I/MiniT2I` repository — request access on the model page and try again.

In [ ]:
#@title Log in to Hugging Face
from huggingface_hub import login, notebook_login, whoami, model_info
from huggingface_hub.utils import RepositoryNotFoundError, HfHubHTTPError

token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    login(token=token, add_to_git_credential=False)
else:
    notebook_login()

print("Logged in as:", whoami().get("name"))

try:
    model_info("MiniT2I/MiniT2I")
    print("Verified Hugging Face model access: MiniT2I/MiniT2I")
except (RepositoryNotFoundError, HfHubHTTPError) as exc:
    raise RuntimeError(
        "Cannot access 'MiniT2I/MiniT2I'. Check that your token has access to the repo, "
        "or update MODEL_ID if the repo was renamed."
    ) from exc

# 3. Choose your settings

These values control the finetuning run. The defaults reproduce a good Naruto-style adapter, so you can run the notebook top to bottom without changing anything.

A few you might want to adjust:
- `MAX_TRAIN_STEPS` — how long to train. Lower it (e.g. `50`) for a quick test, raise it for a stronger effect.
- `RANK` / `LORA_ALPHA` — the size of the LoRA adapter. Larger means more capacity but a bigger file.
- `SAMPLE_PROMPTS` — the prompts generated at the end. Edit these to see your own ideas in the learned style.

`SEED` fixes the randomness so your run is reproducible.

In [ ]:
#@title Finetuning settings
MODEL_ID = "MiniT2I/MiniT2I"  #@param {type:"string"}
MODEL_TYPE = "b16"  #@param ["b16", "l16"]
DATASET_ID = "lambdalabs/naruto-blip-captions"  #@param {type:"string"}
OUTPUT_DIR = Path("minit2i_naruto_lora_demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANK = 32  #@param {type:"integer"}
LORA_ALPHA = 32  #@param {type:"integer"}
LR = 1e-3  #@param {type:"number"}
BATCH_SIZE = 16  #@param {type:"integer"}
GRAD_ACCUM = 1  #@param {type:"integer"}
MAX_TRAIN_STEPS = 400  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

VALIDATION_PROMPTS = [
    "anime portrait of a ninja with orange hair, high quality",
    "a young ninja character in anime style, detailed face, colorful background",
]

SAMPLE_PROMPTS = [
    "female ninja character in Naruto anime style, long black hair, red eyes, clean line art",
    "blue-haired anime ninja in Naruto style, green eyes, detailed face, clean cel shading",
    "female anime ninja in Naruto style, short pink hair, red outfit, clean line art",
    "Naruto anime style, hooded ninja with blue lightning, intense eyes, moonlit training field",
]

random.seed(SEED)
torch.manual_seed(SEED)
print("Output directory:", OUTPUT_DIR.resolve())

# 4. Model definition

So this notebook can run on its own, the next cell defines the MiniT2I model in code. It is the same architecture used to train the released weights, which we load in the following step.

You do not need to read or understand this cell to use the notebook — run it once and feel free to collapse it.

In [ ]:
#@title MiniT2I architecture (run, then collapse)
from dataclasses import dataclass
from typing import Optional

from diffusers import ModelMixin
from diffusers.configuration_utils import ConfigMixin, register_to_config
from diffusers.schedulers.scheduling_utils import SchedulerMixin


def rotate_half(x):
    x1, x2 = x.reshape(*x.shape[:-1], 2, -1).unbind(dim=-2)
    return torch.cat((-x2, x1), dim=-1)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        y = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return y * self.weight


class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.frequency_embedding_size = frequency_embedding_size
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )

    def forward(self, t):
        half = self.frequency_embedding_size // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
        args = t.float()[:, None] * freqs[None]
        emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        return self.mlp(emb.to(dtype=self.mlp[0].weight.dtype))


class BottleneckPatchEmbed(nn.Module):
    def __init__(self, img_size=512, patch_size=16, in_channels=3, pca_channels=128, hidden_size=1248):
        super().__init__()
        self.proj1 = nn.Conv2d(in_channels, pca_channels, kernel_size=patch_size, stride=patch_size, bias=False)
        self.proj2 = nn.Conv2d(pca_channels, hidden_size, kernel_size=1, stride=1, bias=True)

    def forward(self, x):
        x = self.proj2(self.proj1(x))
        return x.flatten(2).transpose(1, 2)


class SwiGLUMlp(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        hidden_dim = (hidden_features + 7) // 8 * 8
        self.w1 = nn.Linear(in_features, hidden_dim, bias=False)
        self.w3 = nn.Linear(in_features, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, in_features, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class TextRotaryEmbedding1D(nn.Module):
    def __init__(self, head_dim, theta=10000.0):
        super().__init__()
        self.head_dim = head_dim
        self.theta = theta

    def forward(self, x):
        b, length, h, d = x.shape
        inv = 1.0 / (self.theta ** (torch.arange(0, d, 2, device=x.device, dtype=torch.float32) / d))
        pos = torch.arange(length, device=x.device, dtype=torch.float32)
        angles = torch.einsum("l,f->lf", pos, inv)
        angles = torch.cat([angles, angles], dim=-1)
        cos = angles.cos().to(dtype=x.dtype)
        sin = angles.sin().to(dtype=x.dtype)
        return x * cos[None, :, None, :] + rotate_half(x) * sin[None, :, None, :]


class VisionRotaryEmbeddingFast(nn.Module):
    def __init__(self, head_dim, theta=10000.0):
        super().__init__()
        self.dim = head_dim // 2
        self.theta = theta

    def forward(self, x):
        length = x.shape[1]
        side = int(math.sqrt(length))
        if side * side != length:
            raise ValueError(f"image token length must be square, got {length}")
        freqs = 1.0 / (self.theta ** (torch.arange(0, self.dim, 2, device=x.device, dtype=torch.float32)[: self.dim // 2] / self.dim))
        t = torch.arange(side, device=x.device, dtype=torch.float32)
        base = torch.einsum("l,f->lf", t, freqs)
        f_h, f_w = torch.broadcast_tensors(base[:, None, :], base[None, :, :])
        angles = torch.cat([f_h, f_w], dim=-1)
        angles = torch.cat([angles, angles], dim=-1).reshape(length, -1)
        cos = angles.cos().to(dtype=x.dtype)
        sin = angles.sin().to(dtype=x.dtype)
        return x * cos[None, :, None, :] + rotate_half(x) * sin[None, :, None, :]


class MultiModalRotaryEmbeddingFast(nn.Module):
    def __init__(self, head_dim):
        super().__init__()
        self.text_rope = TextRotaryEmbedding1D(head_dim)
        self.vision_rope = VisionRotaryEmbeddingFast(head_dim)

    def forward(self, x, txt_len):
        txt = self.text_rope(x[:, :txt_len])
        img = self.vision_rope(x[:, txt_len:])
        return torch.cat([txt, img], dim=1)


class PlainTextTransformerBlock(nn.Module):
    def __init__(self, hidden_size=1248, num_heads=24, head_dim=52, mlp_ratio=2.7):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        inner_dim = num_heads * head_dim
        self.norm1 = RMSNorm(hidden_size)
        self.norm2 = RMSNorm(hidden_size)
        self.qkv = nn.Linear(hidden_size, inner_dim * 3)
        self.attn_proj = nn.Linear(inner_dim, hidden_size)
        self.mlp = SwiGLUMlp(hidden_size, int(hidden_size * mlp_ratio))
        self.q_norm = RMSNorm(head_dim)
        self.k_norm = RMSNorm(head_dim)
        self.rope = TextRotaryEmbedding1D(head_dim)

    def forward(self, txt):
        b, length, _ = txt.shape
        qkv = self.qkv(self.norm1(txt)).reshape(b, length, 3, self.num_heads, self.head_dim)
        q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]
        q = self.rope(self.q_norm(q))
        k = self.rope(self.k_norm(k))
        attn = torch.einsum("bqhd,bkhd->bhqk", q, k) * (self.head_dim ** -0.5)
        out = torch.einsum("bhqk,bkhd->bqhd", attn.softmax(dim=-1), v).reshape(b, length, -1)
        txt = txt + self.attn_proj(out)
        txt = txt + self.mlp(self.norm2(txt))
        return txt


class DoubleStreamDiTBlock(nn.Module):
    def __init__(self, hidden_size=1248, txt_hidden_size=1248, num_heads=24, head_dim=52, mlp_ratio=2.7):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        inner_dim = num_heads * head_dim
        self.img_norm1 = RMSNorm(hidden_size)
        self.img_norm2 = RMSNorm(hidden_size)
        self.txt_norm1 = RMSNorm(txt_hidden_size)
        self.txt_norm2 = RMSNorm(txt_hidden_size)
        self.img_qkv = nn.Linear(hidden_size, inner_dim * 3)
        self.txt_qkv = nn.Linear(txt_hidden_size, inner_dim * 3)
        self.q_norm = RMSNorm(head_dim)
        self.k_norm = RMSNorm(head_dim)
        self.rope = MultiModalRotaryEmbeddingFast(head_dim)
        self.img_attn_proj = nn.Linear(inner_dim, hidden_size)
        self.txt_attn_proj = nn.Linear(inner_dim, txt_hidden_size)
        self.img_mlp = SwiGLUMlp(hidden_size, int(hidden_size * mlp_ratio))
        self.txt_mlp = SwiGLUMlp(txt_hidden_size, int(txt_hidden_size * mlp_ratio))

    def forward(self, x, txt, vec):
        b, li, _ = x.shape
        lt = txt.shape[1]
        qkv_i = self.img_qkv(self.img_norm1(x)).reshape(b, li, 3, self.num_heads, self.head_dim)
        qkv_t = self.txt_qkv(self.txt_norm1(txt)).reshape(b, lt, 3, self.num_heads, self.head_dim)
        q_i, k_i, v_i = qkv_i[:, :, 0], qkv_i[:, :, 1], qkv_i[:, :, 2]
        q_t, k_t, v_t = qkv_t[:, :, 0], qkv_t[:, :, 1], qkv_t[:, :, 2]
        q_i, k_i = self.q_norm(q_i), self.k_norm(k_i)
        q_t, k_t = self.q_norm(q_t), self.k_norm(k_t)
        q = self.rope(torch.cat([q_t, q_i], dim=1), txt_len=lt)
        k = self.rope(torch.cat([k_t, k_i], dim=1), txt_len=lt)
        v = torch.cat([v_t, v_i], dim=1)
        attn = torch.einsum("bqhd,bkhd->bhqk", q, k) * (self.head_dim ** -0.5)
        out = torch.einsum("bhqk,bkhd->bqhd", attn.softmax(dim=-1), v)
        x = x + self.img_attn_proj(out[:, lt:].reshape(b, li, -1))
        txt = txt + self.txt_attn_proj(out[:, :lt].reshape(b, lt, -1))
        x = x + self.img_mlp(self.img_norm2(x))
        txt = txt + self.txt_mlp(self.txt_norm2(txt))
        return x, txt


class FinalLayer(nn.Module):
    def __init__(self, hidden_size=1248, patch_size=16, out_channels=3):
        super().__init__()
        self.norm_final = RMSNorm(hidden_size)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels)

    def forward(self, x, vec=None):
        return self.linear(self.norm_final(x))


def get_1d_sincos_pos_embed(embed_dim, pos):
    omega = torch.arange(embed_dim // 2, device=pos.device, dtype=torch.float32)
    omega = 1.0 / (10000 ** (omega / (embed_dim / 2.0)))
    out = torch.einsum("m,d->md", pos.reshape(-1), omega)
    return torch.cat([out.sin(), out.cos()], dim=1)


def get_2d_sincos_pos_embed(embed_dim, grid_size, device, dtype):
    grid_h = torch.arange(grid_size, device=device, dtype=torch.float32)
    grid_w = torch.arange(grid_size, device=device, dtype=torch.float32)
    grid = torch.meshgrid(grid_w, grid_h, indexing="xy")
    grid = torch.stack(grid, dim=0).reshape(2, 1, grid_size, grid_size)
    emb_h = get_1d_sincos_pos_embed(embed_dim // 2, grid[0])
    emb_w = get_1d_sincos_pos_embed(embed_dim // 2, grid[1])
    return torch.cat([emb_h, emb_w], dim=1).to(dtype=dtype)


@dataclass
class MMJiTConfig:
    image_size: int = 512
    patch_size: int = 16
    in_channels: int = 3
    txt_input_size: int = 1024
    hidden_size: int = 768
    txt_hidden_size: int = 768
    cond_vec_size: int = 768
    depth_double: int = 17
    txt_preamble_depth: int = 2
    num_heads: int = 12
    head_dim: int = 64
    mlp_ratio: float = 2.6667
    pca_channels: int = 128
    prompt_length: int = 256
    n_T: int = 100
    prediction: str = "x"
    sampler: str = "euler"
    cfg_channels: int = 3
    cfg_interval: tuple = (0.0, 1.0)
    llm: str = "google/flan-t5-large"


class MMJiT(nn.Module):
    def __init__(self, cfg: MMJiTConfig):
        super().__init__()
        self.cfg = cfg
        self.latent_img_size = cfg.image_size // cfg.patch_size
        self.img_embedder = BottleneckPatchEmbed(cfg.image_size, cfg.patch_size, cfg.in_channels, cfg.pca_channels, cfg.hidden_size)
        self.txt_embedder = nn.Linear(cfg.txt_input_size, cfg.txt_hidden_size, bias=False)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, cfg.txt_input_size))
        self.t_embedder = TimestepEmbedder(cfg.cond_vec_size)
        self.pooled_embedder = nn.Linear(cfg.txt_input_size, cfg.cond_vec_size, bias=False)
        self.txt_preamble_blocks = nn.ModuleList(
            [PlainTextTransformerBlock(cfg.txt_hidden_size, cfg.num_heads, cfg.head_dim, cfg.mlp_ratio) for _ in range(cfg.txt_preamble_depth)]
        )
        self.double_blocks = nn.ModuleList(
            [DoubleStreamDiTBlock(cfg.hidden_size, cfg.txt_hidden_size, cfg.num_heads, cfg.head_dim, cfg.mlp_ratio) for _ in range(cfg.depth_double)]
        )
        self.final_layer = FinalLayer(cfg.hidden_size, cfg.patch_size, cfg.in_channels)

    def unpatchify(self, x):
        b = x.shape[0]
        p = self.cfg.patch_size
        c = self.cfg.in_channels
        h = w = int(math.sqrt(x.shape[1]))
        x = x.reshape(b, h, w, p, p, c)
        x = torch.einsum("nhwpqc->nchpwq", x)
        return x.reshape(b, c, h * p, w * p)

    def forward(self, img, t, context, attn_mask):
        if img.ndim == 4 and img.shape[1] != self.cfg.in_channels:
            img = img.permute(0, 3, 1, 2)
        attn_mask = attn_mask.to(device=context.device)
        context = torch.where(attn_mask[:, :, None] > 0.5, context, self.mask_token.to(dtype=context.dtype))
        x = self.img_embedder(img)
        pos = get_2d_sincos_pos_embed(self.cfg.hidden_size, self.latent_img_size, x.device, x.dtype)
        x = x + pos[None]
        t_vec = self.t_embedder(t)
        txt = self.txt_embedder(context.to(dtype=self.txt_embedder.weight.dtype))
        pooled_text = context.mean(dim=1)
        vec = t_vec + self.pooled_embedder(pooled_text.to(dtype=self.pooled_embedder.weight.dtype))
        for block in self.txt_preamble_blocks:
            txt = block(txt)
        for block in self.double_blocks:
            x, txt = block(x, txt, vec)
        combined = torch.cat([txt, x], dim=1)
        out = self.final_layer(combined, vec)
        return self.unpatchify(out[:, txt.shape[1]:, :])


class DiffusionModel(nn.Module):
    def __init__(self, cfg: Optional[MMJiTConfig] = None):
        super().__init__()
        self.cfg = cfg or MMJiTConfig()
        self.net = MMJiT(self.cfg)

    def real_t_to_embed_t(self, t):
        return t

    def pred_velocity(self, x, t, text, mask):
        x0 = self.net(x, self.real_t_to_embed_t(t), text, mask)
        return (x0 - x) / torch.clamp(1 - t[:, None, None, None], min=0.05)

    def cfg_velocity(self, x, t, text, mask, cfg_scale):
        b = x.shape[0]
        out = self.pred_velocity(
            torch.cat([x, x], 0), torch.cat([t, t], 0),
            torch.cat([text, text], 0), torch.cat([mask, torch.zeros_like(mask)], 0),
        )
        cond, uncond = out[:b], out[b:]
        use_cfg = ((t >= self.cfg.cfg_interval[0]) & (t <= self.cfg.cfg_interval[1])).to(out.dtype)
        scale = torch.where(use_cfg[:, None, None, None] > 0,
                            torch.tensor(cfg_scale, device=x.device, dtype=out.dtype),
                            torch.tensor(1.0, device=x.device, dtype=out.dtype))
        return uncond + (cond - uncond) * scale

    @torch.no_grad()
    def sample(self, text, mask, cfg_scale=6.0, generator=None, progress=False):
        b = text.shape[0]
        dtype = next(self.parameters()).dtype
        x = torch.randn(b, self.cfg.in_channels, self.cfg.image_size, self.cfg.image_size,
                        generator=generator, device=text.device, dtype=dtype) * 2
        timesteps = torch.linspace(0.0, 1.0, self.cfg.n_T + 1, device=text.device, dtype=dtype)
        iterator = tqdm(range(self.cfg.n_T)) if progress else range(self.cfg.n_T)
        for i in iterator:
            t_cur = timesteps[i].expand(b)
            t_next = timesteps[i + 1].expand(b)
            v = self.cfg_velocity(x, t_cur, text.to(dtype), mask.to(dtype), cfg_scale)
            x = x + (t_next - t_cur)[:, None, None, None] * v
        return x


class MiniT2IFlowMatchScheduler(SchedulerMixin, ConfigMixin):
    config_name = "scheduler_config.json"

    @register_to_config
    def __init__(self, train_t_schedule="lognorm", t_lognorm_mu=-0.8, t_lognorm_sigma=0.8, num_inference_steps=100):
        if train_t_schedule not in {"uniform", "lognorm"}:
            raise ValueError(f"Unsupported train_t_schedule: {train_t_schedule}")

    def sample_train_timesteps(self, batch_size, device, dtype=torch.float32, generator=None):
        if self.config.train_t_schedule == "uniform":
            return torch.rand(batch_size, device=device, dtype=dtype, generator=generator)
        normal = torch.randn(batch_size, device=device, dtype=torch.float32, generator=generator)
        normal = normal * self.config.t_lognorm_sigma + self.config.t_lognorm_mu
        return torch.sigmoid(normal).to(dtype=dtype)


class MiniT2IMMJiTModel(ModelMixin, ConfigMixin):
    config_name = "config.json"

    @register_to_config
    def __init__(self, image_size=512, patch_size=16, in_channels=3, txt_input_size=1024, hidden_size=768,
                 txt_hidden_size=768, cond_vec_size=768, depth_double=17, txt_preamble_depth=2, num_heads=12,
                 head_dim=64, mlp_ratio=2.6666666666666665, pca_channels=128, prompt_length=256, n_T=100,
                 prediction="x", sampler="euler", cfg_channels=3, cfg_interval=(0.0, 1.0), llm="google/flan-t5-large"):
        super().__init__()
        cfg = MMJiTConfig(
            image_size=image_size, patch_size=patch_size, in_channels=in_channels, txt_input_size=txt_input_size,
            hidden_size=hidden_size, txt_hidden_size=txt_hidden_size, cond_vec_size=cond_vec_size,
            depth_double=depth_double, txt_preamble_depth=txt_preamble_depth, num_heads=num_heads, head_dim=head_dim,
            mlp_ratio=mlp_ratio, pca_channels=pca_channels, prompt_length=prompt_length, n_T=n_T, prediction=prediction,
            sampler=sampler, cfg_channels=cfg_channels, cfg_interval=tuple(cfg_interval), llm=llm,
        )
        self.model = DiffusionModel(cfg)

    @property
    def mmjit_config(self):
        return self.model.cfg

    def pred_velocity(self, x, t, text, mask):
        return self.model.pred_velocity(x, t, text, mask)

    def sample(self, text, mask, cfg_scale=6.0, generator=None, progress=False):
        return self.model.sample(text, mask, cfg_scale=cfg_scale, generator=generator, progress=progress)


print("Defined MiniT2I model classes.")

# 5. Download the pretrained model

This downloads the MiniT2I-B/16 weights from Hugging Face and assembles the pieces we need:
- the **image model** that turns prompts into pictures,
- a **scheduler** that defines the noise levels used during training and generation,
- a **tokenizer** and a **text encoder** (FLAN-T5) that convert your prompt text into numbers the model understands.

The text encoder and the base image model are frozen — we will only train the LoRA adapter added in the next step. The download runs once and is cached for the rest of the session.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, T5EncoderModel

MODEL_DIRS = {"b16": "minit2i-b-16", "l16": "minit2i-l-16"}
model_subdir = MODEL_DIRS[MODEL_TYPE]
model_root = Path(snapshot_download(repo_id=MODEL_ID, allow_patterns=[f"{model_subdir}/transformer/*"]))

transformer = MiniT2IMMJiTModel.from_pretrained(model_root / model_subdir / "transformer", torch_dtype=weight_dtype)
scheduler = MiniT2IFlowMatchScheduler(train_t_schedule="lognorm", t_lognorm_mu=-0.8, t_lognorm_sigma=0.8)
cfg = transformer.mmjit_config

tokenizer = AutoTokenizer.from_pretrained(cfg.llm)
text_encoder = T5EncoderModel.from_pretrained(cfg.llm, torch_dtype=torch.float32)
text_encoder.to(device=device, dtype=torch.float32).eval().requires_grad_(False)
transformer.requires_grad_(False)

print("image size:", cfg.image_size, "| prompt length:", cfg.prompt_length, "| text encoder:", cfg.llm)

# 6. Add the LoRA adapter

Here we attach the trainable LoRA adapter to the frozen model. Instead of updating the model's original weights, LoRA inserts small low-rank matrices into the model's linear layers (its attention, feed-forward, and text-projection layers) and trains only those.

The printout shows how few parameters are actually trainable — typically a small fraction of the full model — which is what makes this kind of finetuning fast and lightweight.

In [ ]:
from peft import LoraConfig, get_peft_model

TARGET_KEYWORDS = ("qkv", "attn_proj", "w1", "w2", "w3", "txt_embedder", "pooled_embedder")
target_modules = [
    name for name, module in transformer.named_modules()
    if isinstance(module, nn.Linear) and any(key in name for key in TARGET_KEYWORDS)
]
print("LoRA target modules:", len(target_modules))


def init_lora_weights(model, seed):
    """Kaiming-uniform A and zero B, under a dedicated RNG so init is reproducible."""
    lora_modules = {n: m for n, m in model.named_modules() if hasattr(m, "lora_A") and hasattr(m, "lora_B")}
    gen = torch.Generator().manual_seed(seed)
    saved = torch.random.get_rng_state()
    torch.random.set_rng_state(gen.get_state())
    try:
        for target in target_modules:
            module = next(m for n, m in lora_modules.items() if n.endswith(target))
            nn.init.kaiming_uniform_(module.lora_A["default"].weight, a=math.sqrt(5))
            nn.init.zeros_(module.lora_B["default"].weight)
    finally:
        torch.random.set_rng_state(saved)


lora_config = LoraConfig(r=RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.0, init_lora_weights=True, target_modules=target_modules)
saved_rng = torch.random.get_rng_state()
try:
    transformer = get_peft_model(transformer, lora_config)
    init_lora_weights(transformer, SEED)
finally:
    torch.random.set_rng_state(saved_rng)

transformer.to(device=device, dtype=weight_dtype).train()
trainable_params = [p for p in transformer.parameters() if p.requires_grad]
for p in trainable_params:
    p.data = p.data.float()

minit2i = transformer.base_model.model  # underlying MiniT2IMMJiTModel, LoRA-injected
print("trainable parameters:", sum(p.numel() for p in trainable_params))

# 7. Load the training data

This loads the Naruto image–caption dataset from Hugging Face and prepares the images: each one is resized and cropped to the size MiniT2I expects and scaled to the range the model was trained on. The data is shuffled with a fixed seed so the run is reproducible.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_ID, split="train")

image_transform = transforms.Compose([
    transforms.Resize(cfg.image_size, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(cfg.image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])


class NarutoCaptionDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        image = ImageOps.exif_transpose(row["image"]).convert("RGB")
        return {"pixel_values": image_transform(image), "caption": str(row["text"])}


def collate_fn(examples):
    return {
        "pixel_values": torch.stack([ex["pixel_values"] for ex in examples]),
        "captions": [ex["caption"] for ex in examples],
    }


data_generator = torch.Generator().manual_seed(SEED)
train_dataloader = DataLoader(
    NarutoCaptionDataset(raw_dataset), batch_size=BATCH_SIZE, shuffle=True,
    generator=data_generator, num_workers=2, drop_last=True, collate_fn=collate_fn,
)
print("images:", len(raw_dataset), "| batches per epoch:", len(train_dataloader))

# 8. Training and image-generation helpers

This cell defines the helper functions used below:
- `compute_loss` measures how well the model predicts the training images from noisy versions of them — this is the signal the LoRA adapter learns from.
- `sample_prompt` / `run_validation` generate images from a prompt, which we use to watch progress during training and to produce final samples.
- `show_grid` displays a set of images.

Running this cell just defines the functions; nothing happens yet.

In [ ]:
def encode_prompts(captions):
    tokens = tokenizer(captions, return_tensors="pt", padding="max_length", truncation=True, max_length=cfg.prompt_length)
    input_ids = tokens.input_ids.to(device)
    attention_mask = tokens.attention_mask.to(device)
    with torch.no_grad():
        prompt_embeds = text_encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
    return prompt_embeds.to(dtype=weight_dtype), attention_mask.to(device)


def tensor_to_pil(images):
    images = (images.clamp(-1, 1) * 127.5 + 128.0).clamp(0, 255).to(torch.uint8)
    images = images.permute(0, 2, 3, 1).cpu().numpy()
    return [Image.fromarray(image) for image in images]


train_generator = torch.Generator(device=device).manual_seed(SEED)


def compute_loss(batch):
    images = batch["pixel_values"].to(device=device, dtype=weight_dtype)
    prompt_embeds, attention_mask = encode_prompts(batch["captions"])
    t = scheduler.sample_train_timesteps(images.shape[0], device=device, dtype=weight_dtype, generator=train_generator)
    noise = torch.randn(images.shape, device=device, dtype=weight_dtype, generator=train_generator) * 2
    x_t = images * t[:, None, None, None] + noise * (1 - t[:, None, None, None])
    target = (images - x_t) / torch.clamp(1 - t[:, None, None, None], min=0.05)
    pred = minit2i.pred_velocity(x_t, t, prompt_embeds, attention_mask.to(dtype=weight_dtype))
    return F.mse_loss(pred.float(), target.float(), reduction="mean")


@torch.no_grad()
def sample_prompt(prompt, seed, cfg_scale=6.0, steps=50):
    was_training = transformer.training
    old_steps = minit2i.model.cfg.n_T
    transformer.eval()
    minit2i.model.cfg.n_T = int(steps)
    try:
        prompt_embeds, attention_mask = encode_prompts([prompt])
        generator = torch.Generator(device=device).manual_seed(seed)
        images = minit2i.sample(prompt_embeds, attention_mask.to(dtype=weight_dtype), cfg_scale=cfg_scale, generator=generator)
        return tensor_to_pil(images)[0]
    finally:
        minit2i.model.cfg.n_T = old_steps
        transformer.train(was_training)


def run_validation(step):
    val_dir = OUTPUT_DIR / "validation_images"
    val_dir.mkdir(parents=True, exist_ok=True)
    for prompt_idx, prompt in enumerate(VALIDATION_PROMPTS):
        seed = SEED + prompt_idx
        image = sample_prompt(prompt, seed=seed, steps=50)
        image.save(val_dir / f"step_{step:06d}_prompt_{prompt_idx:02d}_seed_{seed:06d}.png")


def show_grid(paths, cols=4, title=None):
    paths = list(paths)
    if not paths:
        print("No images to show.")
        return
    rows = math.ceil(len(paths) / cols)
    plt.figure(figsize=(4 * cols, 4 * rows))
    if title:
        plt.suptitle(title)
    for i, path in enumerate(paths):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(Image.open(path))
        plt.title(Path(path).name, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# 9. Train the adapter

This is the main training loop. The progress bar shows the running loss, which should trend downward as the adapter learns the style. A set of validation images is saved before training starts (step 0) and every 100 steps, so you can compare the model before and after.

When training finishes, the adapter is saved to disk. This is the longest cell — expect several minutes.

In [ ]:
optimizer = torch.optim.AdamW(trainable_params, lr=LR, betas=(0.9, 0.999), weight_decay=0.0)

run_validation(step=0)

micro_step = 0
global_step = 0
running_loss = None
progress = tqdm(total=MAX_TRAIN_STEPS, desc="LoRA updates")
optimizer.zero_grad(set_to_none=True)

while global_step < MAX_TRAIN_STEPS:
    for batch in train_dataloader:
        loss = compute_loss(batch) / GRAD_ACCUM
        loss.backward()
        micro_step += 1
        if micro_step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            value = float(loss.detach().cpu()) * GRAD_ACCUM
            running_loss = value if running_loss is None else 0.95 * running_loss + 0.05 * value
            progress.update(1)
            progress.set_postfix(loss=f"{value:.4f}", ema=f"{running_loss:.4f}")

            if global_step % 100 == 0:
                run_validation(step=global_step)
            if global_step >= MAX_TRAIN_STEPS:
                break

progress.close()

adapter_dir = OUTPUT_DIR / "adapter"
transformer.save_pretrained(adapter_dir)
print("Finished", global_step, "updates. Saved adapter to", adapter_dir)

# 10. See the progress

These are the validation images saved during training, in order. The earliest ones (step 0) come from the untouched base model; the later ones should look progressively more like the Naruto style as the adapter learns.

In [ ]:
show_grid(sorted((OUTPUT_DIR / "validation_images").glob("*.png")), cols=2, title="Validation images during finetuning")

# 11. Generate your own images

Now use the finetuned adapter to generate fresh images. These come from the `SAMPLE_PROMPTS` you set in the configuration cell — go back and edit them to try your own ideas, then rerun this cell. More inference steps (here, 100) generally give cleaner results.

In [ ]:
sample_dir = OUTPUT_DIR / "samples"
sample_dir.mkdir(parents=True, exist_ok=True)

sample_paths = []
for prompt_idx, prompt in enumerate(SAMPLE_PROMPTS):
    seed = 128 + prompt_idx
    image = sample_prompt(prompt, seed=seed, cfg_scale=6.0, steps=100)
    path = sample_dir / f"prompt_{prompt_idx:02d}_seed_{seed:03d}.png"
    image.save(path)
    sample_paths.append(path)

show_grid(sample_paths, cols=4, title="Samples from the finetuned LoRA")

# What you made

Everything is saved under the `minit2i_naruto_lora_demo/` folder:

- `adapter/` — your trained LoRA adapter. This is small and is all you need to reapply the style later; load it on top of the base MiniT2I model with PEFT.
- `validation_images/` — the progress snapshots from training.
- `samples/` — the final images you generated.

You can download these from the Colab file browser (folder icon in the left sidebar).

Where to go next:
- Edit `SAMPLE_PROMPTS` and rerun the last cell to explore the learned style.
- Swap `DATASET_ID` in the configuration cell for a different image–caption dataset to teach MiniT2I a different style.
- Train longer (raise `MAX_TRAIN_STEPS`) for a stronger effect, or shorter (`50`) for a quick test.